In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

import joblib

import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("../dataset/processed/preprocessed_dataset.csv")

df.head()

,label,message,message_length,clean_message,tokens
0,ham,"Go until jurong point, crazy.. Available only ...",111,go jurong point crazy available bugis n great ...,"['go', 'jurong', 'point', 'crazy', 'available'..."
1,ham,Ok lar... Joking wif u oni...,29,ok lar joking wif u oni,"['ok', 'lar', 'joking', 'wif', 'u', 'oni']"
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155,free entry wkly comp win fa cup final tkts st ...,"['free', 'entry', 'wkly', 'comp', 'win', 'fa',..."
3,ham,U dun say so early hor... U c already then say...,49,u dun say early hor u c already say,"['u', 'dun', 'say', 'early', 'hor', 'u', 'c', ..."
4,ham,"Nah I don't think he goes to usf, he lives aro...",61,nah dont think go usf life around though,"['nah', 'dont', 'think', 'go', 'usf', 'life', ..."


In [3]:
df["message_length"] = df["message"].apply(len)

In [4]:
df["url_count"] = df["message"].str.count(r"http|www")

In [5]:
df["digit_count"] = df["message"].str.count(r"\d")

In [6]:
df["exclamation_count"] = df["message"].str.count("!")

In [7]:
df["uppercase_count"] = df["message"].apply(
    lambda x: sum(1 for c in x if c.isupper())
)

In [8]:
df[
    [
        "message_length",
        "url_count",
        "digit_count",
        "exclamation_count",
        "uppercase_count"
    ]
].head()

,message_length,url_count,digit_count,exclamation_count,uppercase_count
0,111,0,0,0,3
1,29,0,0,0,2
2,155,0,25,0,10
3,49,0,0,0,2
4,61,0,0,0,2


In [9]:
df["label"] = df["label"].map(
    {
        "ham":0,
        "spam":1
    }
)

In [10]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

In [12]:
print(df["clean_message"].isnull().sum())

6


In [13]:
df[df["clean_message"].isnull()]

,label,message,message_length,clean_message,tokens,url_count,digit_count,exclamation_count,uppercase_count
939,0,Where @,7,NaN,[],0,0,0,1
1560,0,645,3,NaN,[],0,3,0,0
2675,0,Can a not?,10,NaN,[],0,0,0,1
3191,0,:),3,NaN,[],0,0,0,0
4276,0,:( but your not here....,24,NaN,[],0,0,0,0
4500,0,:-) :-),7,NaN,[],0,0,0,0


In [14]:
df["clean_message"] = df["clean_message"].fillna("")

In [15]:
df = df[df["clean_message"].str.strip() != ""]
df.reset_index(drop=True, inplace=True)

In [16]:
print(df["clean_message"].isnull().sum())
print((df["clean_message"].str.strip() == "").sum())

0
0


In [17]:
X_text = vectorizer.fit_transform(df["clean_message"])

In [18]:
metadata = df[
    [
        "message_length",
        "url_count",
        "digit_count",
        "exclamation_count",
        "uppercase_count"
    ]
].values

In [19]:
from scipy.sparse import hstack

X = hstack(
    [
        X_text,
        metadata
    ]
)

In [20]:
y = df["label"]

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [22]:
print("Training Samples :", X_train.shape)
print("Testing Samples  :", X_test.shape)

Training Samples : (4130, 5005)
Testing Samples  : (1033, 5005)


In [23]:
joblib.dump(
    vectorizer,
    "../models/tfidf_vectorizer.pkl"
)

['../models/tfidf_vectorizer.pkl']

In [24]:
joblib.dump(X_train, "../models/X_train.pkl")
joblib.dump(X_test, "../models/X_test.pkl")
joblib.dump(y_train, "../models/y_train.pkl")
joblib.dump(y_test, "../models/y_test.pkl")

['../models/y_test.pkl']

In [25]:
print("="*50)
print("Feature Engineering Completed")
print("="*50)

print(f"Training Samples : {X_train.shape[0]}")
print(f"Testing Samples  : {X_test.shape[0]}")
print(f"Features         : {X_train.shape[1]}")

Feature Engineering Completed
Training Samples : 4130
Testing Samples  : 1033
Features         : 5005


In [26]:
df.to_csv(
    "../dataset/processed/final_dataset.csv",
    index=False
)